# CHAY CHINH THUC 30 epoch — P3 noKD More vs Forget-MI (bo do thoi gian moi)

Doi `JOB` o Cell 2 roi Run All. Moi account chay DUNG mot job.

| JOB | Chay gi | Uoc tinh |
|---|---|---|
| `p3more` | P3 noKD More, 30 epoch (phuong phap de xuat) | ~2.5-3h |
| `fmi`    | Forget-MI tai lap, 30 epoch (moc so sanh)     | ~3-3.5h |

**Phai chay CA HAI.** Ban Forget-MI cu khong dung duoc de so thoi gian: no chay
truoc khi co bo do moi (chua tach T_selection, chua do peak theo giai doan) va
truoc khi `pin_memory` duoc dong bo giua hai phuong phap.

**GPU phai giong nhau o ca hai account** (cung chon T4 x2 HOAC cung P100).
Muc 17 cua huong dan cam tron ket qua tu GPU khac nhau — Cell 1 se in ten GPU.

Ket qua thoi gian: `timing_*.json` trong thu muc output cua tung run.
Dung `tools/timing_table.py` (chay o local) de dung bang cho khoa luan.


In [ ]:
# Cell 1: setup + CHOT CHAN code da push
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
_adv=open('training/adv_common.py').read()
assert 'ce_selector' in _adv and 'checkpoint_selection_' in _adv, \
    '❌ adv_common CHUA co hook CE-selector -> chay `git push` code MOI roi moi Save Version!'
assert 'OnlineCESelector' in open('training/ce_selector_pilot.py').read(), '❌ git push code moi truoc!'
assert os.path.exists('training/forgetmi_p3_cand.py'), '❌ chua push forgetmi_p3_cand.py!'
print('✅ Code CE-selector da co (hook + OnlineCESelector).')
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON + path discovery (mimic/iu) + ablation
import glob, os
DATASET    = 'mimic'   # 'mimic' | 'iu'
FORGET_PCT = 3         # 3 | 6 | 10  (iu: 3)
SEED       = 42

# --- DOI DUNG 1 DONG NAY tren moi account ---
JOB    = 'p3more'     # 'p3more' | 'fmi'
EPOCHS = 30
RUN_NO = 1            # 1 = lan chay chinh; dat 2 neu chay lap de bao mean+/-std (muc 17)
assert JOB in ('p3more','fmi')

assert DATASET in ('mimic','iu') and FORGET_PCT in (3,6,10)
if DATASET=='iu': assert FORGET_PCT==3,'IU chi co 3%'

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

tag=f'{DATASET}{FORGET_PCT}per'
OUT=f'/kaggle/working/final_{tag}_s{SEED}'
RESULTS=f'/kaggle/working/results_final_{tag}.csv'

if DATASET=='mimic':
    CONFIG='config_advanced_kaggle.yaml'
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gh=[b for b in bins(MOD) if f'model_retrained_{FORGET_PCT}per' in b]
    GOLD=os.path.dirname(gh[0]) if gh else BASE; HAS_GOLD=bool(gh)
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{FORGET_PCT}per.csv'
else:
    CONFIG='config_loku_iu_kaggle.yaml'
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    RAD=fd('chest-xrays-indiana-university')
    assert DATA and MOD and RAD,'Add forget-mi-data-iu + forget-mi-models-iu + forget-mi-models-iu-re + chest-xrays-indiana-university'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE; HAS_GOLD=bool(reb)
    def first_existing(root, rels):
        for r in rels:
            p=os.path.join(root,r)
            if os.path.exists(p): return p
        return None
    tsv=glob.glob(os.path.join(DATA,'**','all_data.tsv'),recursive=True) or glob.glob('/kaggle/input/**/all_data.tsv',recursive=True)
    TEXT=os.path.dirname(tsv[0]) if tsv else first_existing(DATA,['data/metadata','metadata'])
    IMG=first_existing(DATA,['data/img_data','img_data']) or (first_existing(RAD,['images/images_normalized','images']) if RAD else None) or RAD
    sp=glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True) or glob.glob('/kaggle/input/**/iu-split.csv',recursive=True) or glob.glob(os.path.join(DATA,'**','*iu*split*.csv'),recursive=True)
    fg=glob.glob(os.path.join(DATA,'**',f'forget_set_{FORGET_PCT}per_iu.csv'),recursive=True) or glob.glob(f'/kaggle/input/**/forget_set_{FORGET_PCT}per_iu.csv',recursive=True)
    assert sp and fg,f'Khong thay iu-split / forget_set_iu (glob toan input)'
    SPLIT=sp[0]; FORGET=fg[0]
    if not TEXT or not IMG:
        print('⚠️ TEXT',TEXT,'IMG',IMG,'- liet ke input:')
        for r,d,f in os.walk(DATA):
            if r[len(DATA):].count(os.sep)<=2: print(' ',r,'->',[x for x in f][:4])

for n,p in {'BASE':BASE,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'
COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'results_csv_path':RESULTS,'ce_selector':1,'s4_delta':0.15,
        'use_noise':1}          # Forget-MI Eq.(1)-(2): reference = og(BAN NHIEU) - anh Gaussian + text perturb
print('DATASET',DATASET,'PCT',FORGET_PCT,'| config',CONFIG,'| tag',tag,'| GOLD',HAS_GOLD,'| JOB',JOB)
print('BASE',BASE); print('SPLIT',SPLIT); print('FORGET',FORGET)

# ---------------- CAU HINH JOB ----------------
# P3 noKD More = scheme uni_nokd + mo rong target LoRA (MLP SciBERT + 2 khoi conv cuoi).
# Dung '|' phan tach target vi ',' da bi --override dung de tach cap key=value.
MLP_TXT = 'attention.output.dense|intermediate.dense|output.dense'
MORE    = {'lora_extra_target_modules': MLP_TXT, 'lora_image_last_k_blocks': 2}

RID = f'{JOB}_{tag}_s{SEED}' + ('' if RUN_NO==1 else f'_r{RUN_NO}')
OD  = f'{OUT}/{RID}'
print('JOB',JOB,'| EPOCHS',EPOCHS,'| RUN_NO',RUN_NO,'| run id',RID)
if JOB=='p3more': print('   overrides:',MORE)


In [ ]:
# Cell 3: CHAY (30 epoch). Bo do thoi gian ghi timing_*.json vao OD.
import os, subprocess, time
env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled',
     'PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}

if JOB=='p3more':
    ovr=dict(COMMON); ovr.update(MORE)
    ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,
                'history_csv_path':f'/kaggle/working/perepoch_{RID}.csv'})
    cmd=['python','training/forgetmi_p3_cand.py','--config',CONFIG,'--seed',str(SEED),
         '--scheme','uni_nokd','--fresh','--override',
         ','.join(f'{k}={v}' for k,v in ovr.items())]
else:
    # Forget-MI dung config baseline + key rieng cua no (ce_selector_out thay vi ce_selector).
    ovr={k:v for k,v in COMMON.items() if k not in ('ce_selector','s4_delta')}
    ovr.update({'id':RID,'output_dir':OD,'unlearn_epochs':EPOCHS,
                'results_csv_path':RESULTS,'evaluate_last_and_best':1,
                'ce_selector_out':f'{OD}/checkpoint_selection_forgetmi',
                'history_csv_path':f'/kaggle/working/perepoch_{RID}.csv'})
    cmd=['python','training/forgetmi_partial.py','--config','config_baseline_kaggle.yaml',
         '--seed',str(SEED),'--fresh','--override',
         ','.join(f'{k}={v}' for k,v in ovr.items())]

print('='*72+f'\n{RID}\n'+'='*72)
t0=time.time()
try:
    subprocess.run(cmd,env=env,check=True)
    print(f'OK {RID}  wall {(time.time()-t0)/3600:.2f}h')
except subprocess.CalledProcessError as e:
    print('FAIL',RID,'rc=',e.returncode)


In [ ]:
# Cell 4: eval OG + GOLD tren D_t_final (chi khi CORE; ablation khong can lai)
import os, subprocess
def evalref(label, mpath):
    ovr=dict(COMMON); ovr.pop('ce_selector',None); ovr.pop('s4_delta',None)
    ovr['output_dir']=f'{OUT}/_ref'; ovr['results_csv_path']=RESULTS
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
    cmd=['python','training/forgetmi_eval_only.py','--config',CONFIG,'--seed',str(SEED),
         '--label',label,'--model_type','pretrained','--model_path',mpath,'--method','reference','--override',arg]
    print('eval-ref',label)
    try: subprocess.run(cmd,env=env,check=True)
    except subprocess.CalledProcessError as e: print('FAIL',label,e.returncode)
# Chi can chay tren MOT account (OG/GOLD giong nhau moi candidate). Dat False o 4 acc con lai.
RUN_REF = True
if RUN_REF:
    evalref(f'og_{tag}',BASE)
    if HAS_GOLD: evalref(f're_{tag}',GOLD)
    else: print('(khong co GOLD cho',tag,')')
else:
    print('RUN_REF=False -> bo qua (dung OG/GOLD tu account khac)')


In [ ]:
# Cell 5: doc timing JSON + metric cuoi. TAI CA FILE .json VE de dung bang.
import glob, json, os, pandas as pd
pd.set_option('display.width',200)

js=sorted(glob.glob(f'{OUT}/**/timing_*.json',recursive=True))
if not js:
    print('CHUA co timing_*.json — run that bai hoac chua chay xong.')
for f in js:
    d=json.load(open(f,encoding='utf-8'))
    print('='*70); print(os.path.basename(f))
    print(f"  method     {d.get('method')}   selector {d.get('selector')}   "
          f"ckpt_policy {d.get('checkpoint_policy')}")
    print(f"  GPU        {d.get('gpu_name')}  CUDA {d.get('cuda_version')}  torch {d.get('torch_version')}")
    print(f"  precision  {d.get('precision_mode')}  pin_memory {d.get('pin_memory')}  "
          f"workers {d.get('num_workers')}")
    print(f"  epochs {d.get('epochs')}  updates {d.get('optimizer_updates')}  "
          f"bs {d.get('batch_size')}  lr {d.get('learning_rate')}")
    print(f"  T_fisher   {d.get('fisher_seconds',0):8.1f}s")
    print(f"  T_fila     {d.get('fila_seconds',0):8.1f}s")
    print(f"  T_train    {d.get('train_seconds',0):8.1f}s   (epoch mean "
          f"{d.get('epoch_train_seconds',{}).get('mean',0):.1f}s "
          f"+/- {d.get('epoch_train_seconds',{}).get('std',0):.1f})")
    print(f"  T_core     {d.get('core_seconds',0):8.1f}s   <-- chi so chinh cua RQ3")
    print(f"  T_selection{d.get('selection_seconds',0):8.1f}s   (ckpt I/O {d.get('ckpt_seconds',0):.1f}s)")
    print(f"  T_eval     {d.get('eval_seconds',0):8.1f}s")
    print(f"  T_pipeline {d.get('pipeline_seconds',0):8.1f}s")
    print(f"  chan doan  {d.get('diagnostic_seconds',0):8.1f}s   (ngoai pipeline)")
    print(f"  params     {d.get('trainable_params',0):,} / {d.get('total_params',0):,} "
          f"= {100*d.get('trainable_ratio',0):.2f}%")
    print(f"  peak alloc {d.get('core_peak_allocated_gb',0):.2f} GB   "
          f"reserved {d.get('core_peak_reserved_gb',0):.2f} GB")
    print(f"  theo giai doan: {d.get('peak_allocated_gb')}")

if os.path.exists(RESULTS):
    print('\n===== metric cuoi =====')
    dr=pd.read_csv(RESULTS)
    cols=[c for c in ['id','method','checkpoint_kind','selected_epoch','Forget_AUC',
                      'Forget_Macro_F1','Test_AUC','Test_Macro_F1','MIA','forget_ce',
                      'test_ce','trainable_ratio'] if c in dr.columns]
    print(dr[cols].to_string(index=False))

print('\nTAI VE: timing_*.json (BAT BUOC) + results_final_*.csv + perepoch_*.csv')
print('Sau do chay o LOCAL:')
print('  python tools/timing_table.py --fmi <timing_baseline_partial_*.json> \\')
print('       --p3 <timing_p3cand_*.json> --p3-name "P3-more" --out-tex bang.tex')
